# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## Running this completed notebook
Keep `07-yelp-dataset.txt` in the same folder as this notebook. Run all cells from top to bottom.
If dependencies are missing, run `%pip install numpy pandas scikit-learn torch gensim` in a notebook cell, then restart the kernel and run all cells.
Task 1 uses the mean of known Word2Vec word embeddings; unseen words are ignored and reviews without known words receive a zero vector. Task 2 follows the required architecture and training settings.
The preprocessing indentation, stratified split, and gradient clearing in the original example have been corrected. Training has not been executed in the preparation environment, so accuracy will be computed when you run the notebook.


## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [ ]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

In [ ]:
print(df["label"].value_counts())

## 2. Preprocessing the Dataset

In [ ]:
import re

def preprocess_text(text):
    # Lowercase, remove punctuation, and normalize whitespace.
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

# Apply preprocessing outside the function, after its definition.
df["review"] = df["review"].apply(preprocess_text)
df.head()


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)


## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [ ]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [ ]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [ ]:
epochs = 10
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()  # Clear gradients before each backward pass.
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [ ]:
import numpy as np
from gensim.models import Word2Vec

# The reviews have already been cleaned. Preserve their split order.
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

# Fit only on the training reviews to avoid test-data leakage.
word2vec = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=42
)

def review_vector(tokens, word2vec_model):
    # Average known word embeddings to obtain one vector per review.
    known_words = [word for word in tokens if word in word2vec_model.wv]
    if not known_words:
        # Empty reviews or reviews containing only unseen words.
        return np.zeros(word2vec_model.vector_size, dtype=np.float32)
    return word2vec_model.wv[known_words].mean(axis=0).astype(np.float32)

X_train_w2v = np.stack([review_vector(tokens, word2vec) for tokens in train_tokens])
X_test_w2v = np.stack([review_vector(tokens, word2vec) for tokens in test_tokens])

X_train_w2v_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_w2v_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)
y_train_w2v_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test_w2v_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

print("Training Word2Vec tensor shape:", X_train_w2v_tensor.shape)
print("Testing Word2Vec tensor shape:", X_test_w2v_tensor.shape)


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [ ]:
torch.manual_seed(42)

# Input: 200 features; hidden layers: 128, 64, 32; output: one logit.
w2v_model = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
print(w2v_model)

w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(w2v_model.parameters(), lr=0.001)
w2v_epochs = 15

for epoch in range(w2v_epochs):
    w2v_model.train()
    w2v_optimizer.zero_grad()
    logits = w2v_model(X_train_w2v_tensor)
    loss = w2v_loss_function(logits, y_train_w2v_tensor)
    loss.backward()
    w2v_optimizer.step()
    print(f"Epoch {epoch + 1}/{w2v_epochs} - Loss: {loss.item():.4f}")

w2v_model.eval()
with torch.no_grad():
    w2v_test_logits = w2v_model(X_test_w2v_tensor)
    w2v_probabilities = torch.sigmoid(w2v_test_logits)
    w2v_predictions = (w2v_probabilities >= 0.5).int().numpy().ravel()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Word2Vec Test Accuracy: {w2v_accuracy:.3f}")
print("Classification Report:")
print(classification_report(
    y_test, w2v_predictions, labels=[0, 1],
    target_names=["Negative", "Positive"], zero_division=0
))
print("Confusion Matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, w2v_predictions, labels=[0, 1]))
